# Feature Selection Pipeline (v2)

Dataset mới từ Leads_Engineering.ipynb: **21 features**, đã split Train/Test.

**Quan trọng:** Feature selection chỉ thực hiện trên **Train**, sau đó áp dụng kết quả lên **Test**.

| Bước | Phương pháp | Loại |
|------|-------------|------|
| 1 | Loại features phương sai thấp | Filter |
| 2 | Loại features trùng lặp | Filter |
| 3 | Mutual Information + Loại đa cộng tuyến | Filter |
| 4 | Embedded Selection (Gradient Boosting) | Embedded |
| 5 | RFECV (Recursive Feature Elimination) | Wrapper |
| 6 | Đánh giá kết quả | Evaluation |

In [18]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, SelectFromModel, RFECV
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [19]:
df_train = pd.read_csv('Leads_Engineered_Scaled_Train.csv')
df_test = pd.read_csv('Leads_Engineered_Scaled_Test.csv')

X_train = df_train.drop(columns=['Converted'])
y_train = df_train['Converted']
X_test = df_test.drop(columns=['Converted'])
y_test = df_test['Converted']

print(f'Train: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test:  {X_test.shape[0]} samples, {X_test.shape[1]} features')
print(f'\nTarget distribution (Train):\n{y_train.value_counts(normalize=True).round(3)}')
print(f'\nDanh sách features:')
for i, col in enumerate(X_train.columns, 1):
    print(f'  {i}. {col} (dtype={X_train[col].dtype})')

Train: 6152 samples, 21 features
Test:  1538 samples, 21 features

Target distribution (Train):
Converted
0    0.601
1    0.399
Name: proportion, dtype: float64

Danh sách features:
  1. Lead Origin (dtype=int64)
  2. Lead Source (dtype=int64)
  3. Do Not Email (dtype=int64)
  4. Do Not Call (dtype=int64)
  5. TotalVisits (dtype=float64)
  6. Total Time Spent on Website (dtype=float64)
  7. Page Views Per Visit (dtype=float64)
  8. Last Activity (dtype=int64)
  9. Specialization (dtype=int64)
  10. What is your current occupation (dtype=int64)
  11. Lead Profile (dtype=int64)
  12. City (dtype=int64)
  13. A free copy of Mastering The Interview (dtype=int64)
  14. Last Notable Activity (dtype=int64)
  15. ttspow_pvps (dtype=float64)
  16. tttspow_totalvisits (dtype=float64)
  17. pvps_totalvisits (dtype=float64)
  18. Time_Per_Page (dtype=float64)
  19. Time_Per_Visit (dtype=float64)
  20. Engagement_Score (dtype=float64)
  21. Engagement_Level (dtype=float64)


## Bước 1: Loại features phương sai thấp
Nếu 1 giá trị chiếm trên 99% dữ liệu → Loại bỏ

In [20]:
n_before = X_train.shape[1]
threshold = 0.99
cols_to_drop = [col for col in X_train.columns
                if X_train[col].value_counts(normalize=True, dropna=False).values[0] >= threshold]

X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

print(f'Trước: {n_before} → Còn lại: {X_train.shape[1]} (xóa {len(cols_to_drop)})')
if cols_to_drop:
    print(f'Các cột bị xóa: {cols_to_drop}')

Trước: 21 → Còn lại: 20 (xóa 1)
Các cột bị xóa: ['Do Not Call']


## Bước 2: Loại features trùng lặp hoàn toàn

In [21]:
n_before = X_train.shape[1]
duplicated_features = X_train.columns[X_train.T.duplicated()].tolist()

X_train = X_train.drop(columns=duplicated_features)
X_test = X_test.drop(columns=duplicated_features)

print(f'Trước: {n_before} → Còn lại: {X_train.shape[1]} (xóa {len(duplicated_features)})')
if duplicated_features:
    print(f'Các cột bị xóa: {duplicated_features}')

Trước: 20 → Còn lại: 20 (xóa 0)


## Bước 3: Mutual Information & Loại đa cộng tuyến
- Dùng **Mutual Information** để đánh giá mối quan hệ feature-target
- Khi 2 features tương quan > 0.7 với nhau, giữ feature có MI score cao hơn

In [22]:
# Tiền xử lý nếu cần
X_clean = X_train.copy()
for col in X_clean.select_dtypes(include=['object']).columns:
    X_clean[col] = LabelEncoder().fit_transform(X_clean[col].astype(str))
X_clean = X_clean.fillna(X_clean.median())

# Tính Mutual Information
print('Đang tính Mutual Information...')
mi_scores = mutual_info_classif(X_clean, y_train, random_state=42)
mi_series = pd.Series(mi_scores, index=X_train.columns)

print(f'\nMI scores cho tất cả features:')
print(mi_series.sort_values(ascending=False).round(4))
print(f'\nSố features có MI = 0: {(mi_series == 0).sum()}')

Đang tính Mutual Information...

MI scores cho tất cả features:
Engagement_Score                          0.1537
Total Time Spent on Website               0.1476
tttspow_totalvisits                       0.1401
Time_Per_Page                             0.1234
ttspow_pvps                               0.1196
Time_Per_Visit                            0.1165
Lead Profile                              0.0886
What is your current occupation           0.0799
Last Activity                             0.0659
Last Notable Activity                     0.0518
Lead Origin                               0.0365
pvps_totalvisits                          0.0260
Lead Source                               0.0255
Engagement_Level                          0.0216
Specialization                            0.0214
TotalVisits                               0.0162
Page Views Per Visit                      0.0125
Do Not Email                              0.0065
A free copy of Mastering The Interview    0.0000
City 

In [23]:
# Loại đa cộng tuyến
print('Đang tính ma trận tương quan...')
corr_matrix = X_clean.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

cols_to_drop_corr = set()
threshold_corr = 0.7

for col in upper_tri.columns:
    highly_correlated = upper_tri.index[upper_tri[col] > threshold_corr].tolist()
    for corr_col in highly_correlated:
        # Dùng MI để quyết định giữ feature nào
        if mi_series.get(col, 0) > mi_series.get(corr_col, 0):
            cols_to_drop_corr.add(corr_col)
        else:
            cols_to_drop_corr.add(col)

n_before = X_train.shape[1]
X_train = X_train.drop(columns=list(cols_to_drop_corr))
X_test = X_test.drop(columns=list(cols_to_drop_corr))

print(f'Trước: {n_before} → Còn lại: {X_train.shape[1]} (xóa {len(cols_to_drop_corr)})')
if cols_to_drop_corr:
    print(f'Các cột bị xóa: {sorted(cols_to_drop_corr)}')

Đang tính ma trận tương quan...
Trước: 20 → Còn lại: 12 (xóa 8)
Các cột bị xóa: ['Last Notable Activity', 'Page Views Per Visit', 'Time_Per_Page', 'Time_Per_Visit', 'Total Time Spent on Website', 'TotalVisits', 'ttspow_pvps', 'tttspow_totalvisits']


## Bước 4: Embedded Feature Selection (Gradient Boosting)
Dùng GBM importance, giữ features có importance >= **0.5 × mean** (ngưỡng thấp hơn để giữ nhiều features hơn cho RFECV).

In [24]:
X_emb = X_train.copy()
for col in X_emb.select_dtypes(include=['object']).columns:
    X_emb[col] = LabelEncoder().fit_transform(X_emb[col].astype(str))
X_emb = X_emb.fillna(X_emb.median())

print(f'Training GradientBoosting trên {X_emb.shape[1]} features...')
gbc = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.1,
    subsample=0.8, random_state=42
)
gbc.fit(X_emb, y_train)

importances = pd.Series(gbc.feature_importances_, index=X_train.columns)
print(f'\nFeature importance:')
print(importances.sort_values(ascending=False).round(4))

# SelectFromModel: giữ features có importance >= 0.5 * mean
threshold_val = 0.5 * importances.mean()
selector = SelectFromModel(gbc, prefit=True, threshold=threshold_val)
mask = selector.get_support()
selected_cols = X_train.columns[mask].tolist()

n_before = X_train.shape[1]
X_train = X_train[selected_cols]
X_test = X_test[selected_cols]

print(f'\nMean importance: {importances.mean():.4f}')
print(f'Threshold (0.5 * mean): {threshold_val:.4f}')
print(f'Trước: {n_before} → Sau Embedded Selection: {X_train.shape[1]} features')
print(f'Features được giữ: {selected_cols}')

Training GradientBoosting trên 12 features...

Feature importance:
Engagement_Score                          0.4531
Lead Profile                              0.1240
What is your current occupation           0.1167
Last Activity                             0.0958
Lead Origin                               0.0822
pvps_totalvisits                          0.0697
Specialization                            0.0198
Do Not Email                              0.0150
Lead Source                               0.0134
City                                      0.0071
Engagement_Level                          0.0017
A free copy of Mastering The Interview    0.0016
dtype: float64

Mean importance: 0.0833
Threshold (0.5 * mean): 0.0417
Trước: 12 → Sau Embedded Selection: 6 features
Features được giữ: ['Lead Origin', 'Last Activity', 'What is your current occupation', 'Lead Profile', 'pvps_totalvisits', 'Engagement_Score']


## Bước 5: RFECV — Tìm số features tối ưu
Thuật toán tự động chọn số features tối ưu dựa trên ROC-AUC cross-validation.

In [25]:
X_rfe = X_train.copy()
for col in X_rfe.select_dtypes(include=['object']).columns:
    X_rfe[col] = LabelEncoder().fit_transform(X_rfe[col].astype(str))
X_rfe = X_rfe.fillna(X_rfe.median())

print(f'Chạy RFECV trên {X_rfe.shape[1]} features...')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_estimator = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=42, n_jobs=-1
)

rfecv = RFECV(
    estimator=rf_estimator,
    step=1,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1,
    min_features_to_select=3
)
rfecv.fit(X_rfe, y_train)

selected_mask = rfecv.support_
selected_features_final = X_train.columns[selected_mask].tolist()

X_train_final = X_train[selected_features_final]
X_test_final = X_test[selected_features_final]

print(f'\n=== Kết quả RFECV ===')
print(f'Số features tối ưu: {rfecv.n_features_}')
print(f'\nDanh sách features được chọn:')
for i, f in enumerate(selected_features_final, 1):
    print(f'  {i}. {f}')

Chạy RFECV trên 6 features...

=== Kết quả RFECV ===
Số features tối ưu: 6

Danh sách features được chọn:
  1. Lead Origin
  2. Last Activity
  3. What is your current occupation
  4. Lead Profile
  5. pvps_totalvisits
  6. Engagement_Score


## Bước 6: Đánh giá kết quả
Đánh giá trên cả **Cross-Validation (Train)** và **Test set**.

In [26]:
from sklearn.metrics import roc_auc_score

# Chuẩn bị data
X_eval_train = X_train_final.copy()
for col in X_eval_train.select_dtypes(include=['object']).columns:
    X_eval_train[col] = LabelEncoder().fit_transform(X_eval_train[col].astype(str))
X_eval_train = X_eval_train.fillna(X_eval_train.median())

X_eval_test = X_test_final.copy()
for col in X_eval_test.select_dtypes(include=['object']).columns:
    X_eval_test[col] = LabelEncoder().fit_transform(X_eval_test[col].astype(str))
X_eval_test = X_eval_test.fillna(X_eval_test.median())

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Cross-Validation trên Train ---
rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42, n_jobs=-1)
rf_cv = cross_val_score(rf, X_eval_train, y_train, cv=cv, scoring='roc_auc')

gbc_eval = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42
)
gbc_cv = cross_val_score(gbc_eval, X_eval_train, y_train, cv=cv, scoring='roc_auc')

# --- Đánh giá trên Test set ---
rf.fit(X_eval_train, y_train)
rf_test_auc = roc_auc_score(y_test, rf.predict_proba(X_eval_test)[:, 1])

gbc_eval.fit(X_eval_train, y_train)
gbc_test_auc = roc_auc_score(y_test, gbc_eval.predict_proba(X_eval_test)[:, 1])

print(f'=== Kết quả với {X_train_final.shape[1]} features ===')
print(f'')
print(f'Cross-Validation (Train):')
print(f'  RandomForest AUC:       {rf_cv.mean():.4f} +/- {rf_cv.std():.4f}')
print(f'  GradientBoosting AUC:   {gbc_cv.mean():.4f} +/- {gbc_cv.std():.4f}')
print(f'')
print(f'Test Set:')
print(f'  RandomForest AUC:       {rf_test_auc:.4f}')
print(f'  GradientBoosting AUC:   {gbc_test_auc:.4f}')

=== Kết quả với 6 features ===

Cross-Validation (Train):
  RandomForest AUC:       0.8933 +/- 0.0058
  GradientBoosting AUC:   0.8972 +/- 0.0063

Test Set:
  RandomForest AUC:       0.8970
  GradientBoosting AUC:   0.8989


In [27]:
# Lưu kết quả
df_train_selected = X_train_final.copy()
df_train_selected['Converted'] = y_train.values
df_train_selected.to_csv('Leads_Selected_Train.csv', index=False)

df_test_selected = X_test_final.copy()
df_test_selected['Converted'] = y_test.values
df_test_selected.to_csv('Leads_Selected_Test.csv', index=False)

with open('selected_features_v2.txt', 'w', encoding='utf-8') as f:
    for feat in selected_features_final:
        f.write(feat + '\n')

print(f'Đã lưu Train ({X_train_final.shape[1]} features) vào Leads_Selected_Train.csv')
print(f'Đã lưu Test ({X_test_final.shape[1]} features) vào Leads_Selected_Test.csv')
print(f'Đã lưu danh sách features vào selected_features_v2.txt')

Đã lưu Train (6 features) vào Leads_Selected_Train.csv
Đã lưu Test (6 features) vào Leads_Selected_Test.csv
Đã lưu danh sách features vào selected_features_v2.txt
